# Topic-Hard Split Experiment (Notebook First)

This notebook upgrades the baseline clustering pipeline to:

1. Build topic vectors with **TF-IDF -> TruncatedSVD -> L2 normalization**.
2. Evaluate multiple `k` values (`[20, 30, 40, 50, 60, 80]`) using quality and balance diagnostics.
3. Choose a more usable clustering.
4. Simulate whole-cluster assignment to train/dev/test with a greedy objective.
5. Print split quality summaries **without writing files by default**.

Use this notebook to validate the method before creating an actual split file.

In [1]:
from __future__ import annotations

import json
import math
from dataclasses import dataclass
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.decomposition import TruncatedSVD
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import silhouette_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import Normalizer


In [2]:
# Paths + experiment config
if Path.cwd().name == "notebooks":
    PROJECT_ROOT = Path.cwd().parent
else:
    PROJECT_ROOT = Path.cwd()

CLEANED_PATH = PROJECT_ROOT / "artifacts" / "data" / "cleaned.jsonl"
STANDARD_SPLIT_PATH = PROJECT_ROOT / "artifacts" / "splits" / "standard.json"
TOPIC_HARD_SPLIT_PATH = PROJECT_ROOT / "artifacts" / "splits" / "topic_hard.json"

RANDOM_STATE = 42
K_VALUES = [20, 30, 40, 50, 60, 80]
TARGET_RATIOS = {"train": 0.8, "dev": 0.1, "test": 0.1}

# Safety switch: keep False during experimentation.
SAVE_SPLIT = True

df = pd.read_json(CLEANED_PATH, lines=True, encoding="utf-8")
df = df[["id", "label", "text"]].dropna(subset=["text"]).reset_index(drop=True)
df["label"] = df["label"].astype(int)

print(f"Loaded {len(df):,} rows")
df.head()

Loaded 28,503 rows


,id,label,text
0,sar_000001,1,thirtysomething scientists unveil doomsday clo...
1,sar_000002,0,dem rep. totally nails why congress is falling...
2,sar_000003,0,eat your veggies: 9 deliciously different recipes
3,sar_000004,1,inclement weather prevents liar from getting t...
4,sar_000005,1,mother comes pretty close to using word 'strea...


In [3]:
def build_topic_vectors(
    texts: pd.Series,
    max_features: int = 20_000,
    ngram_range: tuple[int, int] = (1, 2),
    min_df: int = 2,
    max_df: float = 0.9,
    n_components: int = 200,
    random_state: int = 42,
):
    """Build TF-IDF and reduced dense topic vectors."""
    vectorizer = TfidfVectorizer(
        lowercase=True,
        stop_words="english",
        ngram_range=ngram_range,
        max_features=max_features,
        min_df=min_df,
        max_df=max_df,
    )
    x_tfidf = vectorizer.fit_transform(texts)

    max_valid_components = max(2, min(x_tfidf.shape[0] - 1, x_tfidf.shape[1] - 1))
    use_components = min(n_components, max_valid_components)

    svd = TruncatedSVD(n_components=use_components, random_state=random_state)
    normalizer = Normalizer(copy=False)
    reducer = make_pipeline(svd, normalizer)
    x_topic = reducer.fit_transform(x_tfidf)
    return x_tfidf, x_topic, vectorizer, reducer


def normalized_entropy(counts: np.ndarray) -> float:
    probs = counts / counts.sum()
    nonzero = probs[probs > 0]
    entropy = -float(np.sum(nonzero * np.log(nonzero)))
    max_entropy = math.log(len(counts))
    return 0.0 if max_entropy == 0 else entropy / max_entropy


def evaluate_k_values(
    x_topic: np.ndarray,
    labels_true: pd.Series,
    k_values: list[int],
    random_state: int = 42,
    silhouette_sample_size: int = 5_000,
):
    """Return metrics table + predicted cluster labels for each k."""
    rows = []
    labels_by_k: dict[int, np.ndarray] = {}
    global_pos_ratio = float(labels_true.mean())

    sample_size = min(silhouette_sample_size, x_topic.shape[0])

    for k in k_values:
        model = KMeans(n_clusters=k, random_state=random_state, n_init=20)
        cluster_labels = model.fit_predict(x_topic)
        labels_by_k[k] = cluster_labels

        sil = silhouette_score(
            x_topic,
            cluster_labels,
            sample_size=sample_size,
            random_state=random_state,
        )

        counts = pd.Series(cluster_labels).value_counts().sort_index()
        count_values = counts.to_numpy()

        tmp = pd.DataFrame({"cluster": cluster_labels, "label": labels_true})
        per_cluster_pos = tmp.groupby("cluster")["label"].mean()
        mean_abs_label_skew = float((per_cluster_pos - global_pos_ratio).abs().mean())

        rows.append(
            {
                "k": k,
                "inertia": float(model.inertia_),
                "silhouette": float(sil),
                "max_cluster_pct": float(count_values.max() / len(labels_true)),
                "min_cluster_pct": float(count_values.min() / len(labels_true)),
                "entropy_norm": float(normalized_entropy(count_values)),
                "mean_abs_label_skew": mean_abs_label_skew,
            }
        )

    metrics = pd.DataFrame(rows).sort_values("k").reset_index(drop=True)
    return metrics, labels_by_k


def choose_usable_k(metrics: pd.DataFrame) -> int:
    """Composite score: higher silhouette/entropy, lower concentration/skew."""
    m = metrics.copy()
    m["usable_score"] = (
        1.0 * m["silhouette"]
        + 0.20 * m["entropy_norm"]
        - 0.25 * m["max_cluster_pct"]
        - 0.15 * m["mean_abs_label_skew"]
    )
    return int(m.sort_values("usable_score", ascending=False).iloc[0]["k"])


def summarize_clusters(df_local: pd.DataFrame):
    counts = df_local["cluster"].value_counts().sort_index()
    size_table = pd.DataFrame({
        "n": counts,
        "pct": (counts / len(df_local) * 100).round(2),
    })

    label_mix = pd.crosstab(df_local["cluster"], df_local["label"], normalize="index").round(3)
    label_mix.columns = [f"label_{int(c)}" for c in label_mix.columns]
    return size_table, label_mix


def top_terms_by_cluster(x_tfidf, cluster_labels, vectorizer, top_n: int = 12) -> pd.DataFrame:
    """Approximate topic words by average TF-IDF value per cluster."""
    feature_names = np.array(vectorizer.get_feature_names_out())
    rows = []

    for c in sorted(np.unique(cluster_labels)):
        mask = cluster_labels == c
        mean_vec = np.asarray(x_tfidf[mask].mean(axis=0)).ravel()
        top_idx = np.argsort(mean_vec)[::-1][:top_n]
        rows.append({
            "cluster": int(c),
            "top_terms": ", ".join(feature_names[top_idx]),
        })

    return pd.DataFrame(rows)


@dataclass
class SplitState:
    n: int = 0
    pos: int = 0


def assign_clusters_greedily(
    df_local: pd.DataFrame,
    target_ratios: dict[str, float],
    random_state: int = 42,
):
    """
    Assign whole clusters to splits using proportional fill.

    The least-filled split (relative to its target) gets the next cluster.
    Overflow and label-gap terms act as secondary corrections.
    """
    total = sum(target_ratios.values())
    if abs(total - 1.0) > 1e-8:
        raise ValueError(f"Split ratios must sum to 1.0, got {total}")

    total_n = len(df_local)
    global_pos_ratio = float(df_local["label"].mean())
    target_n = {name: int(round(total_n * r)) for name, r in target_ratios.items()}

    cluster_stats = (
        df_local.groupby("cluster")
        .agg(n=("id", "size"), pos=("label", "sum"))
        .reset_index()
        .sort_values("n", ascending=False)
        .reset_index(drop=True)
    )

    rng = np.random.default_rng(random_state)
    cluster_stats["tie_jitter"] = rng.random(len(cluster_stats))
    cluster_stats = cluster_stats.sort_values(["n", "tie_jitter"], ascending=[False, True])

    states = {name: SplitState() for name in target_ratios}
    split_names = list(target_ratios.keys())
    cluster_to_split: dict[int, str] = {}

    w_fill, w_overflow, w_label = 1.0, 2.0, 0.4

    def score(split_name: str, c_n: int, c_pos: int) -> float:
        st = states[split_name]
        new_n = st.n + c_n
        new_pos = st.pos + c_pos

        current_fill = st.n / target_n[split_name] if target_n[split_name] > 0 else float("inf")
        overflow = max(0.0, (new_n - target_n[split_name]) / target_n[split_name])

        new_pos_ratio = (new_pos / new_n) if new_n > 0 else global_pos_ratio
        label_gap = abs(new_pos_ratio - global_pos_ratio)

        return w_fill * current_fill + w_overflow * overflow + w_label * label_gap

    for row in cluster_stats.itertuples(index=False):
        c = int(row.cluster)
        c_n = int(row.n)
        c_pos = int(row.pos)

        best_split = min(split_names, key=lambda s: score(s, c_n, c_pos))
        cluster_to_split[c] = best_split
        states[best_split].n += c_n
        states[best_split].pos += c_pos

    assigned = df_local.copy()
    assigned["split"] = assigned["cluster"].map(cluster_to_split)
    return assigned, cluster_to_split


def summarize_split_preview(df_assigned: pd.DataFrame, cluster_to_split: dict[int, str]) -> None:
    print("=== Dry-run split summary (no file write) ===")
    global_pos = float(df_assigned["label"].mean())
    print(f"Global label_1 ratio: {global_pos:.3f}")

    for split_name in ["train", "dev", "test"]:
        part = df_assigned[df_assigned["split"] == split_name]
        n = len(part)
        label_dist = part["label"].value_counts(normalize=True).sort_index()
        l0 = float(label_dist.get(0, 0.0))
        l1 = float(label_dist.get(1, 0.0))
        clusters = sorted(part["cluster"].unique().tolist())
        print(f"\n{split_name.upper()} ({n} rows)")
        print(f"  label_0: {l0:.3f}")
        print(f"  label_1: {l1:.3f}")
        print(f"  clusters: {clusters}")

    split_cluster = pd.crosstab(df_assigned["split"], df_assigned["cluster"], normalize="index").round(3)
    print("\nSplit x Cluster (row-normalized)")
    display(split_cluster)


In [4]:
x_tfidf, x_topic, vectorizer, reducer = build_topic_vectors(
    df["text"],
    max_features=20_000,
    ngram_range=(1, 2),
    min_df=2,
    max_df=0.9,
    n_components=200,
    random_state=RANDOM_STATE,
)

print(f"TF-IDF shape: {x_tfidf.shape}")
print(f"Topic vector shape: {x_topic.shape}")

metrics_df, labels_by_k = evaluate_k_values(
    x_topic=x_topic,
    labels_true=df["label"],
    k_values=K_VALUES,
    random_state=RANDOM_STATE,
    silhouette_sample_size=5000,
)

metrics_df["usable_score"] = (
    1.0 * metrics_df["silhouette"]
    + 0.20 * metrics_df["entropy_norm"]
    - 0.25 * metrics_df["max_cluster_pct"]
    - 0.15 * metrics_df["mean_abs_label_skew"]
)

metrics_df.sort_values("usable_score", ascending=False).reset_index(drop=True)

TF-IDF shape: (28503, 20000)
Topic vector shape: (28503, 200)


,k,inertia,silhouette,max_cluster_pct,min_cluster_pct,entropy_norm,mean_abs_label_skew,usable_score
0,80,18988.843581,0.136313,0.093780,0.003508,0.916936,0.144173,0.274629
1,60,20292.141094,0.116189,0.152440,0.004421,0.869711,0.162932,0.227581
2,50,21117.130076,0.106508,0.151914,0.003193,0.872925,0.162835,0.218690
3,40,21911.072203,0.094609,0.229906,0.006140,0.850259,0.158216,0.183452
4,30,22804.933311,0.081971,0.296004,0.006876,0.814650,0.167849,0.145723
5,20,23926.908321,0.066081,0.449567,0.009367,0.687895,0.187521,0.063140


In [5]:
chosen_k = choose_usable_k(metrics_df)
print(f"Chosen k: {chosen_k}")

df_clustered = df.copy()
df_clustered["cluster"] = labels_by_k[chosen_k]

df_clustered[["id", "label", "cluster", "text"]].head()

Chosen k: 80


,id,label,cluster,text
0,sar_000001,1,72,thirtysomething scientists unveil doomsday clo...
1,sar_000002,0,55,dem rep. totally nails why congress is falling...
2,sar_000003,0,5,eat your veggies: 9 deliciously different recipes
3,sar_000004,1,5,inclement weather prevents liar from getting t...
4,sar_000005,1,5,mother comes pretty close to using word 'strea...


In [6]:
cluster_sizes_df, cluster_label_mix_df = summarize_clusters(df_clustered)
print("Cluster sizes")
display(cluster_sizes_df)

print("Label mix per cluster")
display(cluster_label_mix_df)

Cluster sizes


,n,pct
cluster,,
0,156,0.55
1,1485,5.21
2,307,1.08
3,1248,4.38
4,267,0.94
...,...,...
75,956,3.35
76,163,0.57
77,138,0.48


Label mix per cluster


,label_0,label_1
cluster,,
0,0.628,0.372
1,0.484,0.516
2,0.502,0.498
3,0.579,0.421
4,0.760,0.240
...,...,...
75,0.547,0.453
76,0.301,0.699
77,0.558,0.442


In [7]:
top_terms_df = top_terms_by_cluster(
    x_tfidf=x_tfidf,
    cluster_labels=df_clustered["cluster"].to_numpy(),
    vectorizer=vectorizer,
    top_n=12,
)
top_terms_df

,cluster,top_terms
0,0,"election, voters, 2016 election, election day,..."
1,1,"million, tv, california, living, super, announ..."
2,2,"like, look like, look, looks like, looks, just..."
3,3,"wedding, film, taylor, king, reveals, swift, t..."
4,4,"women, women march, march, men, girls, women g..."
...,...,...
75,75,"finally, story, senate, girlfriend, today, can..."
76,76,"bush, jeb, jeb bush, george, george bush, chen..."
77,77,"media, social, social media, onion, onion soci..."
78,78,"new, york, new york, unveils, unveils new, rel..."


In [8]:
df_dryrun, cluster_to_split = assign_clusters_greedily(
    df_local=df_clustered,
    target_ratios=TARGET_RATIOS,
    random_state=RANDOM_STATE,
)

summarize_split_preview(df_dryrun, cluster_to_split)

split_sizes = df_dryrun["split"].value_counts().reindex(["train", "dev", "test"])
split_sizes

=== Dry-run split summary (no file write) ===
Global label_1 ratio: 0.475

TRAIN (22875 rows)
  label_0: 0.515
  label_1: 0.485
  clusters: [2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 13, 14, 15, 17, 19, 20, 21, 22, 23, 24, 25, 27, 28, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 42, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 55, 56, 57, 58, 59, 60, 61, 62, 63, 65, 66, 68, 70, 71, 72, 73, 74, 75, 76, 77, 78, 79]

DEV (2774 rows)
  label_0: 0.588
  label_1: 0.412
  clusters: [0, 16, 26, 29, 41, 43]

TEST (2854 rows)
  label_0: 0.540
  label_1: 0.460
  clusters: [1, 12, 18, 54, 64, 67, 69]

Split x Cluster (row-normalized)


cluster,0,1,2,3,4,5,6,7,8,9,...,70,71,72,73,74,75,76,77,78,79
split,,,,,,,,,,,,,,,,,,,,,
dev,0.056,0.00,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,...,0.000,0.00,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000
test,0.000,0.52,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,...,0.000,0.00,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000
train,0.000,0.00,0.013,0.055,0.012,0.117,0.014,0.013,0.006,0.007,...,0.009,0.01,0.033,0.007,0.006,0.042,0.007,0.006,0.039,0.009


split
train    22875
dev       2774
test      2854
Name: count, dtype: int64

In [9]:
# Optional write step (disabled by default).
if SAVE_SPLIT:
    split_obj = {
        "meta": {
            "method": "topic_hard_tfidf_svd_kmeans",
            "ratios": TARGET_RATIOS,
            "random_state": RANDOM_STATE,
            "total_rows": int(len(df_dryrun)),
            "global_label_ratio": {
                "label_0": float((df_dryrun["label"] == 0).mean()),
                "label_1": float((df_dryrun["label"] == 1).mean()),
            },
            "selected_k": int(chosen_k),
            "k_metrics": metrics_df.to_dict(orient="records"),
            "cluster_to_split": {str(k): v for k, v in sorted(cluster_to_split.items())},
        },
        "train": sorted(df_dryrun.loc[df_dryrun["split"] == "train", "id"].tolist()),
        "dev": sorted(df_dryrun.loc[df_dryrun["split"] == "dev", "id"].tolist()),
        "test": sorted(df_dryrun.loc[df_dryrun["split"] == "test", "id"].tolist()),
    }

    TOPIC_HARD_SPLIT_PATH.parent.mkdir(parents=True, exist_ok=True)
    TOPIC_HARD_SPLIT_PATH.write_text(json.dumps(split_obj, indent=2), encoding="utf-8")
    print(f"Saved split -> {TOPIC_HARD_SPLIT_PATH}")
else:
    print("SAVE_SPLIT=False: dry-run only, no split file written.")

Saved split -> e:\CS4248_Project\artifacts\splits\topic_hard.json
